# 03 - Train U-Net + MaxViT-base (issue #20)

**Why.** MaxViT is the primary source model: a transformer encoder that sees the whole chip at once, where the ResNet50 baseline (#19) looks through small windows. Its checkpoint is what the NISAR transfer experiments start from.

**What.** The same training script and config as the CNN arm (`configs/unet_resnet50.yaml`), with three deliberate differences, all recorded in `configs/unet_maxvit.yaml`:
- encoder `maxvit_base` instead of `resnet50`
- `batch_size: 4` with `accum_steps: 2`, an effective batch of 8 like the ResNet run, because batch 8 does not fit on a T4 (trial run: out of memory)
- the output folder

**How.** Run the sections in order on a **T4 GPU** runtime (Runtime -> Change runtime type). Expect about 1.5-2 hours for the full run. If the session disconnects, rerun sections 1-3 and then the resume cell.

## 1. Code and packages

In [ ]:
BRANCH = "20-train-maxvit-source-model"   # change to "main" once the PR has been merged

%cd /content
!rm -rf nisar-flood-mapping
!git clone -b {BRANCH} https://github.com/cs59300-deep-learning/nisar-flood-mapping.git
%cd nisar-flood-mapping
!git --no-pager log --oneline -3
!pip install -q segmentation-models-pytorch timm rasterio pyyaml tqdm matplotlib requests
!nvidia-smi | head -12

## 2. Data (Sen1Floods11 hand-labelled split, about 700 MB, about 4 minutes)

In [ ]:
!PYTHONPATH=src python -m nisar_flood.datasets.sen1floods11 download
!PYTHONPATH=src python -m nisar_flood.datasets.sen1floods11 check

## 3. Output folder on Google Drive

Colab's disk is wiped when a session ends, so the run writes to Drive. The checkpoint is large (about 1.5 GB), so it is saved every 10 epochs (`save_every`), which changes how often it is written, not the training itself. A temporary copy of the config is used so the repo file stays unchanged.

In [ ]:
from google.colab import drive
import os, yaml
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/nisar_runs/unet_maxvit'
os.makedirs(OUT, exist_ok=True)

cfg = yaml.safe_load(open('configs/unet_maxvit.yaml'))
cfg['output']['dir'] = OUT
cfg['train']['save_every'] = 10
yaml.safe_dump(cfg, open('/content/run_maxvit.yaml', 'w'))
print(cfg['data'], cfg['train'], cfg['output'], sep='\n')

## 4. Smoke test (optional, about 6 minutes)

Checks gradient accumulation and `--resume` before the long run: train 3 epochs with a throwaway seed, stop the cell once the `epoch 2/3` bar appears (the epoch-1 checkpoint is saved by then), then resume. The history should list epochs 1, 2, 3 once each. This writes to the local disk, not to Drive.

In [ ]:
!PYTHONPATH=src python -m nisar_flood.training.train --config configs/unet_maxvit.yaml --epochs 3 --seed 99

In [ ]:
!PYTHONPATH=src python -m nisar_flood.training.train --config configs/unet_maxvit.yaml --epochs 3 --seed 99 --resume
!cut -d, -f1-4 runs/unet_maxvit/seed99/history.csv

## 5. Full run (100 epochs, seed 0)

In [ ]:
!PYTHONPATH=src python -m nisar_flood.training.train --config /content/run_maxvit.yaml

**If the session disconnects:** rerun sections 1-3, then run this cell. It continues from the last saved epoch (at most 10 epochs are lost). It refuses to resume if the config differs from the one stored in the checkpoint.

In [ ]:
!PYTHONPATH=src python -m nisar_flood.training.train --config /content/run_maxvit.yaml --resume

## 6. Results

In [ ]:
import json
s = json.load(open(f'{OUT}/seed0/summary.json'))
print({k: v for k, v in s.items() if k != 'config'})
!tail -n 3 {OUT}/seed0/history.csv
from IPython.display import Image
Image(f'{OUT}/seed0/curves.png')

Test and Bolivia scores use the evaluation entry point from #19. This cell has not yet been run for the MaxViT checkpoint, so check its output before reporting numbers.

In [ ]:
!PYTHONPATH=src python -m nisar_flood.cli evaluate --checkpoint {OUT}/seed0/best.pt